# Juliet 최종 평가: Router 정책, End-to-End 탐지, 패치

Frozen test split은 설정 변경에 사용하지 않습니다. 먼저 Expert×Model test outcome matrix를 한 번 수집하고, 그 동일 행렬로 모든 Router 정책을 비교합니다. 이후 선택적으로 실제 Router 경로를 다시 호출해 탐지 결과를 보존하고, 원본 Juliet 패키지의 임시 복사본에서 패치를 compile/test 검증합니다.

In [ ]:
from pathlib import Path
EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
TEST_CASE_LIMIT = 0  # 0 = frozen test split 전체
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
EXECUTE_PAID = False
MAX_REQUESTS_PER_RUN = 500
MAX_USD_PER_RUN = 20.0
RESERVE_USD_PER_REQUEST = 0.10
RUN_LIVE_DETECTION = False
LIVE_CASE_LIMIT = 100
RUN_PATCH_EVALUATION = False
PATCH_FINDING_LIMIT = 50
PATCH_VERIFICATION_COMMANDS = [
    {'name': 'juliet-build', 'command': ['make'], 'timeout_seconds': 300}
]  # 환경에 맞게 수정. 실제 compile/test 명령이 반드시 필요합니다.


In [ ]:
import json, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, evaluate_utility_router
from model_evaluation.live_evaluation import run_live_detection, run_patch_evaluation
from model_evaluation.adapters.llm_security import activate_parent_package
activate_parent_package()
from llm_security.routing import BudgetedUtilityRouter
config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
RUN_DIR = EVAL_ROOT / 'work' / 'router_evaluation'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_evaluation'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
if not ARTIFACT.exists():
    print('먼저 train.ipynb에서 Router artifact를 생성해야 합니다:', ARTIFACT)

## 1. Frozen test case와 candidate cache 생성

In [ ]:
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases', splits=('test',),
    limits={'test': TEST_CASE_LIMIT}, progress=print,
)
candidate_summary = cache_candidates(
    RUN_DIR / 'cases' / 'cases_test.jsonl', RUN_DIR / 'candidates' / 'candidates_test.jsonl',
    max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
)
print(json.dumps({'materialization': materialization, 'candidates': candidate_summary}, ensure_ascii=False, indent=2))

## 2. Artifact의 동일 assignment pool로 test API dry-run

In [ ]:
if ARTIFACT.exists():
    router = BudgetedUtilityRouter.load(ARTIFACT)
    models = sorted({item.model_id for item in router.assignments.values()})
    plan = plan_outcome_matrix(
        cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
        selection_manifest=RUN_DIR / 'selections' / 'selected_test.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / 'outcomes_test.jsonl', model_ids=models,
        max_candidates_per_case=MAX_CANDIDATES_PER_CASE, hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
    print(json.dumps(plan, ensure_ascii=False, indent=2))
else:
    models = []

## 3. Test Expert×Model outcome matrix 수집 (재개 가능)

In [ ]:
if EXECUTE_PAID and ARTIFACT.exists():
    collection = collect_outcome_matrix(
        env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / 'outcomes_test.jsonl',
        ledger_path=RUN_DIR / 'ledgers' / 'test_api_ledger.jsonl', model_ids=models,
        execute_paid=True, max_requests=MAX_REQUESTS_PER_RUN, max_usd=MAX_USD_PER_RUN,
        reserve_usd_per_request=RESERVE_USD_PER_REQUEST,
        max_candidates_per_case=MAX_CANDIDATES_PER_CASE, hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
    print(json.dumps(collection, ensure_ascii=False, indent=2))
else:
    print('PAID LOCKED 또는 artifact 없음: dry-run만 수행했습니다.')

## 4. 최종 정책 비교 + 전체 단계별 End-to-End 지표 (추가 API 없음)

In [ ]:
test_outcomes = RUN_DIR / 'outcomes' / 'outcomes_test.jsonl'
if ARTIFACT.exists() and test_outcomes.exists():
    matrix_audit = audit_outcome_matrix(test_outcomes, expected_assignment_ids=list(router.assignments), selection_manifest=RUN_DIR / 'selections' / 'selected_test.jsonl')
    print(json.dumps(matrix_audit, ensure_ascii=False, indent=2))
    if matrix_audit['complete']:
        final_report = evaluate_utility_router(
            artifact_path=ARTIFACT, test_outcomes=test_outcomes,
            test_cases=RUN_DIR / 'cases' / 'cases_test.jsonl',
            candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
            selection_manifest=RUN_DIR / 'selections' / 'selected_test.jsonl',
            report_path=RESULT_DIR / 'policy_and_end_to_end.json',
            candidate_gate_enabled=False, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        )
        print(json.dumps(final_report, ensure_ascii=False, indent=2))
    else:
        print('평가 보류: test matrix 수집을 재개하세요.')

## 5. 실제 Router 경로 탐지 재실행 (선택)

Outcome replay가 최종 비교의 주 평가입니다. 이 단계는 배포 경로와 같은 Router 선택을 실제 API로 다시 실행해 finding JSON을 보존하는 smoke/end-to-end 검증입니다.

In [ ]:
if EXECUTE_PAID and RUN_LIVE_DETECTION and ARTIFACT.exists():
    live_report = run_live_detection(
        env_file=ENV_FILE, artifact_path=ARTIFACT,
        cases_path=RUN_DIR / 'cases' / 'cases_test.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / 'candidates_test.jsonl',
        output_path=RUN_DIR / 'live' / 'detections.jsonl', ledger_path=RUN_DIR / 'ledgers' / 'live_api_ledger.jsonl',
        execute_paid=True, max_requests=MAX_REQUESTS_PER_RUN, max_usd=MAX_USD_PER_RUN,
        reserve_usd_per_request=RESERVE_USD_PER_REQUEST, max_cases=LIVE_CASE_LIMIT,
        max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
    )
    print(json.dumps(live_report, ensure_ascii=False, indent=2))
else:
    print('Live detection은 기본 비활성화 상태입니다.')

## 6. 실제 패치 생성 및 임시 복사본 검증 (선택)

`PATCH_VERIFICATION_COMMANDS`는 현재 환경에서 성공하는 실제 build/test 명령으로 설정해야 합니다. 원본 `D:\llm-data`는 수정하지 않습니다. `git apply`와 모든 명령이 성공해야 verified repair로 집계됩니다.

In [ ]:
detections = RUN_DIR / 'live' / 'detections.jsonl'
if EXECUTE_PAID and RUN_PATCH_EVALUATION and detections.exists():
    patch_report = run_patch_evaluation(
        env_file=ENV_FILE, detection_path=detections,
        output_path=RUN_DIR / 'live' / 'patches.jsonl', ledger_path=RUN_DIR / 'ledgers' / 'patch_api_ledger.jsonl',
        commands=PATCH_VERIFICATION_COMMANDS, execute_paid=True,
        max_requests=MAX_REQUESTS_PER_RUN, max_usd=MAX_USD_PER_RUN,
        reserve_usd_per_request=RESERVE_USD_PER_REQUEST, max_findings=PATCH_FINDING_LIMIT,
        only_ground_truth_matched=True,
    )
    print(json.dumps(patch_report, ensure_ascii=False, indent=2))
else:
    print('Patch evaluation은 기본 비활성화 상태입니다.')